# 04 FeatureEngineering

Notebook นี้ใช้สำหรับสร้างฟีเจอร์จากข้อมูล timestamp ที่ผ่านการ clean แล้ว


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parents[0]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.utils.paths import INTERIM_DATA_DIR

sns.set_theme(style='whitegrid')
INTERIM_DATA_DIR

WindowsPath('D:/ML/data/interim')

## Load Clean Data

โหลดไฟล์ `vw_timestamp_dashboard_clean.csv` ซึ่งเป็นข้อมูลที่ผ่านการ clean แล้วมาใช้งาน


In [ ]:
source_path = INTERIM_DATA_DIR / 'vw_timestamp_dashboard_clean.csv'
df = pd.read_csv(source_path, encoding='utf-8-sig')

print(f'source: {source_path}')
print(f'shape: {df.shape}')
df.head()

source: D:\ML\data\interim\vw_timestamp_dashboard_clean.csv
shape: (7519, 37)


,PlantName,PickListType,PickDate,TruckSeqNo,CarType,CarNo,PackListNo,CustomerName,QueueTime,PrepareForward,...,PRESTIGEFittingSapAmount,NEUSTILEFittingSapAmount,DURAFittingSapAmount,ACCESSORIESSapAmount,TileStart,TileEnd,FittingStart,FittingEnd,AccStart,AccEnd
0,SB1,Walk in,2026-04-10 08:52:23,1,4 เธฅเนเธญ,84-5388,SB1PL260410015,เธซเธเธ.เธชเธณเธฃเธงเธขเน€เธเธฃเธฒเธกเธดเธ,2026-04-10 08:52:32,N,...,0,0,0,0,2026-04-10 09:23:51,2026-04-10 09:24:56,2026-04-10 09:00:14,2026-04-10 09:04:32,NaN,NaN
1,SB1,Walk in,2026-04-10 08:48:55,4,6 เธฅเนเธญ,70-4573,SB1PL260410014,เธเธฃเธดเธฉเธฑเธ— เธงเธฑเธเนเธฎเธก เธงเธฑเธชเธ”เธธเธเนเธญเธชเธฃเนเธฒเธ เธเธณเธเธฑเธ”,2026-04-10 08:49:00,N,...,0,0,0,0,2026-04-10 09:08:03,2026-04-10 09:13:52,NaN,NaN,NaN,NaN
2,SB1,Walk in,2026-04-10 08:47:25,1,4 เธฅเนเธญ,3เธ’เธญ9423,SB1PL260410013,เธ.เธฃเธงเธกเธเธตเน€เธกเธเธ•เน99 เธเธ.,2026-04-10 08:47:34,N,...,50,0,0,1,2026-04-10 09:15:01,2026-04-10 09:22:14,2026-04-10 08:55:17,2026-04-10 08:58:33,2026-04-10 08:50:45,2026-04-10 08:52:05
3,SB1,Walk in,2026-04-10 08:42:51,2,4 เธฅเนเธญ,เธเธฅ4874,SB1PL260410012,เธ.เน€เธเธดเธเธ—เธญเธเธกเธฒเธงเธฑเธชเธ”เธธ เธเธ.,2026-04-10 08:42:56,N,...,30,0,0,0,2026-04-10 08:55:58,2026-04-10 08:56:39,2026-04-10 08:52:42,2026-04-10 08:54:33,NaN,NaN
4,SB1,Walk in,2026-04-10 08:29:57,6,6 เธฅเนเธญ,71-0658,SB1PL260410011,เธเธฃเธดเธฉเธฑเธ— เธงเธฑเธเนเธฎเธก เธงเธฑเธชเธ”เธธเธเนเธญเธชเธฃเนเธฒเธ เธเธณเธเธฑเธ”,2026-04-10 08:30:02,N,...,0,0,0,0,2026-04-10 08:45:24,2026-04-10 08:46:40,NaN,NaN,NaN,NaN


## Convert Timestamp Columns

แปลงคอลัมน์วันที่และเวลาที่เกี่ยวข้องให้เป็นชนิด `datetime` เพื่อให้สามารถคำนวณต่อได้ถูกต้อง


In [ ]:
timestamp_columns = [
    'PickDate',
    'QueueTime',
    'OperatorCarConfirm',
    'CarConfirm',
    'FirstPostPallet',
    'LastPostPallet',
    'PostingTime',
]

df_feature = df.copy()

for col in timestamp_columns:
    df_feature[col] = pd.to_datetime(df_feature[col], errors='coerce')

df_feature[timestamp_columns].dtypes

PickDate              datetime64[us]
QueueTime             datetime64[us]
OperatorCarConfirm    datetime64[us]
CarConfirm            datetime64[us]
FirstPostPallet       datetime64[us]
LastPostPallet        datetime64[us]
PostingTime           datetime64[us]
dtype: object

## Create Calendar Features

สร้างฟีเจอร์เชิงปฏิทินจาก `OperatorCarConfirm` เพื่อให้โมเดลมองเห็น pattern ตามช่วงเวลาได้ดีขึ้น

- `hour`: ชั่วโมงของ `OperatorCarConfirm`
- `day_of_week`: วันในสัปดาห์ของ `OperatorCarConfirm` โดย `0=Monday` และ `6=Sunday`
- `is_weekend`: ถ้าเป็นวันเสาร์/อาทิตย์ให้เป็น `1` ถ้าเป็นวันทำงานให้เป็น `0`
- `month`: เดือนของ `OperatorCarConfirm`
- `week_of_month`: สัปดาห์ที่เท่าไรของเดือน คำนวณด้วย `((day - 1) // 7) + 1`
- `shift`: แบ่งช่วงเวลาในแต่ละวันตาม `hour` เป็น `night`, `morning`, `afternoon`, `evening`


In [ ]:
calendar_datetime = df_feature['OperatorCarConfirm']

df_feature['hour'] = calendar_datetime.dt.hour
df_feature['day_of_week'] = calendar_datetime.dt.dayofweek
df_feature['is_weekend'] = df_feature['day_of_week'].isin([5, 6]).astype(int)
df_feature['month'] = calendar_datetime.dt.month
df_feature['week_of_month'] = ((calendar_datetime.dt.day - 1) // 7 + 1).astype('Int64')

df_feature['shift'] = pd.cut(
    df_feature['hour'],
    bins=[-1, 5, 11, 17, 23],
    labels=['night', 'morning', 'afternoon', 'evening'],
)

calendar_feature_columns = [
    'OperatorCarConfirm',
    # 'PickDate',
    'hour',
    'day_of_week',
    'is_weekend',
    'month',
    'shift',
    'week_of_month',
]

df_feature[calendar_feature_columns].head()

,OperatorCarConfirm,hour,day_of_week,is_weekend,month,shift,week_of_month
0,2026-04-10 08:52:28,8,4,0,4,morning,2
1,2026-04-10 08:48:56,8,4,0,4,morning,2
2,2026-04-10 08:47:31,8,4,0,4,morning,2
3,2026-04-10 08:42:54,8,4,0,4,morning,2
4,2026-04-10 08:29:59,8,4,0,4,morning,2


## Create Queue Features

Create queue features from `OperatorCarConfirm` by counting active previous trucks that had not reached `PostingTime` yet when the current truck reached `OperatorCarConfirm`.

- `active_queue_before`: previous trucks still active at the current truck's `OperatorCarConfirm`
- `active_queue_before_same_location`: active previous trucks in the same `PostLocationName`


In [ ]:
queue_time_column = 'OperatorCarConfirm'
queue_close_column = 'PostingTime'
queue_date_column = '_queue_date_from_operator_confirm'

queue_feature_columns = [
    'active_queue_before',
    'active_queue_before_same_location',
]

df_feature[queue_feature_columns] = pd.NA
df_feature[queue_date_column] = df_feature[queue_time_column].dt.normalize()

valid_queue_mask = df_feature[queue_time_column].notna()
queue_sort_columns = [queue_time_column]

for optional_sort_column in ['PackListNo', 'TruckSeqNo']:
    if optional_sort_column in df_feature.columns:
        queue_sort_columns.append(optional_sort_column)

queue_ordered = df_feature.loc[valid_queue_mask].sort_values(
    queue_sort_columns,
    kind='mergesort',
)
# Active previous trucks: entered before the current truck and had not posted out yet.
# If a previous truck has missing PostingTime, treat it as still active.
for current_index, current_row in queue_ordered.iterrows():
    current_time = current_row[queue_time_column]
    current_date = current_row[queue_date_column]

    active_before_mask = (
        (queue_ordered[queue_date_column] == current_date)
        & (queue_ordered[queue_time_column] < current_time)
        & (
            queue_ordered[queue_close_column].isna()
            | (queue_ordered[queue_close_column] > current_time)
        )
    )
    active_before = queue_ordered.loc[active_before_mask]

    df_feature.at[current_index, 'active_queue_before'] = len(active_before)
    df_feature.at[current_index, 'active_queue_before_same_location'] = (
        active_before['PostLocationName'].eq(current_row['PostLocationName']).sum()
    )

df_feature[queue_feature_columns] = df_feature[queue_feature_columns].astype('Int64')
df_feature = df_feature.drop(columns=[queue_date_column])

df_feature.sort_values(queue_time_column)[[
    'OperatorCarConfirm',
    'PostingTime',
    'PlantName',
    'PostLocationName',
    *queue_feature_columns,
]].head(10)

,OperatorCarConfirm,PostingTime,PlantName,PostLocationName,active_queue_before,active_queue_before_same_location
7518,2025-11-10 07:28:56,2025-11-10 08:15:03,SB1,เธฅเธฒเธเธเนเธฒเธข3 เธเนเธญเธเธเนเธฒเธข 2 (เธเธฃเธญเธ),0,0
7517,2025-11-10 07:29:09,2025-11-10 08:16:59,SB1,เธฅเธฒเธเธเนเธฒเธข3 เธเนเธญเธเธเนเธฒเธข 2 (เธเธฃเธญเธ),1,1
7516,2025-11-10 07:29:27,2025-11-10 08:16:16,SB1,เธฅเธฒเธเธเนเธฒเธข 2 เธเนเธญเธเธเนเธฒเธข 2 (Prestige),2,0
7515,2025-11-10 07:29:40,2025-11-10 07:54:53,SB1,เธฅเธฒเธเธเนเธฒเธข 2 เธเนเธญเธเธเนเธฒเธข 2 (Prestige),3,1
7514,2025-11-10 07:29:51,2025-11-10 08:18:50,SB1,เธฅเธฒเธเธเนเธฒเธข 2 เธเนเธญเธเธเนเธฒเธข 2 (Prestige),4,2
7513,2025-11-10 07:30:02,2025-11-10 08:21:38,SB1,เธฅเธฒเธเธเนเธฒเธข3 เธเนเธญเธเธเนเธฒเธข 1 (เธเธฃเธญเธ),5,0
7512,2025-11-10 07:30:24,2025-11-10 08:10:26,SB1,เธฅเธฒเธเธเนเธฒเธข 2 เธเนเธญเธเธเนเธฒเธข 1 (Prestige),6,0
7511,2025-11-10 07:30:37,2025-11-10 08:21:32,SB1,เธฅเธฒเธเธเนเธฒเธข3 เธเนเธญเธเธเนเธฒเธข 2 (เธเธฃเธญเธ),7,2
7510,2025-11-10 07:30:47,2025-11-10 08:19:22,SB1,เธฅเธฒเธเธเนเธฒเธข 2 เธเนเธญเธเธเนเธฒเธข 1 (Prestige),8,1
7509,2025-11-10 07:30:58,2025-11-10 07:57:17,SB1,เธฅเธฒเธเธเนเธฒเธข3 เธเนเธญเธเธเนเธฒเธข 2 (เธเธฃเธญเธ),9,3


## Create Target Time Features

สร้างตัวแปรเป้าหมายเชิงเวลา (นาที) จากความต่างของ timestamp ในแต่ละช่วงงาน และสรุปเป็นตัวแปรรวมสำหรับวิเคราะห์เพิ่มเติม

- `wait_call_min`: `CarConfirm` - `OperatorCarConfirm`
- `prepare_loading_min`: `FirstPostPallet` - `CarConfirm`
- `loading_time_min`: `LastPostPallet` - `FirstPostPallet`
- `close_job_min`: `PostingTime` - `LastPostPallet`
- `total_time_min`: ผลรวมของ target time ทั้ง 4 ช่วงข้างต้น


In [ ]:
df_feature['wait_call_min'] = (
    df_feature['CarConfirm'] - df_feature['OperatorCarConfirm']
).dt.total_seconds() / 60

df_feature['prepare_loading_min'] = (
    df_feature['FirstPostPallet'] - df_feature['CarConfirm']
).dt.total_seconds() / 60

df_feature['loading_time_min'] = (
    df_feature['LastPostPallet'] - df_feature['FirstPostPallet']
).dt.total_seconds() / 60

df_feature['close_job_min'] = (
    df_feature['PostingTime'] - df_feature['LastPostPallet']
).dt.total_seconds() / 60

df_feature['total_time_min'] = (
    df_feature['wait_call_min']
    + df_feature['prepare_loading_min']
    + df_feature['loading_time_min']
    + df_feature['close_job_min']
)

time_columns = [
    'wait_call_min',
    'prepare_loading_min',
    'loading_time_min',
    'close_job_min',
    'total_time_min',
]

df_feature[time_columns] = df_feature[time_columns].round(2)

df_feature[time_columns].head()

,wait_call_min,prepare_loading_min,loading_time_min,close_job_min,total_time_min
0,15.63,1.40,12.02,24.33,53.38
1,7.95,2.72,6.23,10.80,27.70
2,9.62,13.63,7.10,19.32,49.67
3,1.58,10.43,15.42,8.12,35.55
4,5.40,4.67,9.88,13.87,33.82


## Create Product Features

สร้างฟีเจอร์จากกลุ่มสินค้า โดยแปลงยอด SAP ของแต่ละกลุ่มเป็นตัวแปรเชิงบูลีน และสร้างตัวแปรสรุปเพื่อใช้วิเคราะห์ผลกระทบของสินค้า

- `has_tile`: ถ้ามียอดสินค้ากลุ่ม Tile มากกว่า 0 ให้เป็น `1` ถ้าไม่มียอดให้เป็น `0`
- `has_fitting`: ถ้ามียอดสินค้ากลุ่ม Fitting มากกว่า 0 ให้เป็น `1` ถ้าไม่มียอดให้เป็น `0`
- `has_accessories`: ถ้า `ACCESSORIESSapAmount` มากกว่า 0 ให้เป็น `1` ถ้าไม่มียอดให้เป็น `0`
- `product_group_count`: ผลรวมของ `has_tile`, `has_fitting`, `has_accessories`
- `total_sap_amount`: ผลรวมยอด SAP ของสินค้าทุกกลุ่ม


In [ ]:
amount_columns = [
    'CPACTileSapAmount',
    'PRESTIGETileSapAmount',
    'NEUSTILETileSapAmount',
    'CPACFittingSapAmount',
    'PRESTIGEFittingSapAmount',
    'NEUSTILEFittingSapAmount',
    'DURAFittingSapAmount',
    'ACCESSORIESSapAmount',
]

df_feature[amount_columns] = df_feature[amount_columns].apply(pd.to_numeric, errors='coerce').fillna(0)

tile_columns = [
    'CPACTileSapAmount',
    'PRESTIGETileSapAmount',
    'NEUSTILETileSapAmount',
]

fitting_columns = [
    'CPACFittingSapAmount',
    'PRESTIGEFittingSapAmount',
    'NEUSTILEFittingSapAmount',
    'DURAFittingSapAmount',
]

df_feature['has_tile'] = (df_feature[tile_columns].sum(axis=1) > 0).astype(int)
df_feature['has_fitting'] = (df_feature[fitting_columns].sum(axis=1) > 0).astype(int)
df_feature['has_accessories'] = (df_feature['ACCESSORIESSapAmount'] > 0).astype(int)
df_feature['product_group_count'] = df_feature[
    ['has_tile', 'has_fitting', 'has_accessories']
].sum(axis=1)

df_feature['total_sap_amount'] = df_feature[amount_columns].sum(axis=1)

product_feature_columns = [
    'has_tile',
    'has_fitting',
    'has_accessories',
    'product_group_count',
    'total_sap_amount',
]

df_feature[product_feature_columns].head()

,has_tile,has_fitting,has_accessories,product_group_count,total_sap_amount
0,1,1,0,2,1647.0
1,1,0,0,1,1600.0
2,1,1,1,3,571.0
3,1,1,0,2,290.0
4,1,0,0,1,1600.0


## Check Created Features

ตรวจสอบค่าเบื้องต้นของฟีเจอร์ที่สร้างขึ้น เช่น สถิติของ target time และตัวอย่างข้อมูลบางส่วน เพื่อยืนยันว่าค่าที่คำนวณได้สมเหตุสมผล


In [ ]:
target_columns = [
    'wait_call_min',
    'prepare_loading_min',
    'loading_time_min',
    'close_job_min',
    'total_time_min',
]

df_feature[target_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
wait_call_min,7519.0,9.440718,9.950396,0.05,2.85,6.18,12.370,88.73
prepare_loading_min,7519.0,10.453413,11.745884,0.10,2.55,6.72,14.570,301.08
loading_time_min,7519.0,27.080614,20.663704,0.02,11.77,22.13,36.665,213.98
close_job_min,7519.0,15.575765,16.980239,1.18,7.00,11.65,19.270,339.90
total_time_min,7519.0,62.550456,28.112211,7.23,43.65,58.67,77.960,385.07


In [ ]:
df_feature[[
    'OperatorCarConfirm',
    'CarConfirm',
    'FirstPostPallet',
    'LastPostPallet',
    'PostingTime',
    'has_tile',
    'has_fitting',
    'has_accessories',
    'product_group_count',
    'total_sap_amount',
    'hour',
    'day_of_week',
    'is_weekend',
    'month',
    'shift',
    'week_of_month',
    'active_queue_before',
    'active_queue_before_same_location',
    'wait_call_min',
    'prepare_loading_min',
    'loading_time_min',
    'close_job_min',
    'total_time_min',
]].head(10)

,OperatorCarConfirm,CarConfirm,FirstPostPallet,LastPostPallet,PostingTime,has_tile,has_fitting,has_accessories,product_group_count,total_sap_amount,...,month,shift,week_of_month,active_queue_before,active_queue_before_same_location,wait_call_min,prepare_loading_min,loading_time_min,close_job_min,total_time_min
0,2026-04-10 08:52:28,2026-04-10 09:08:06,2026-04-10 09:09:30,2026-04-10 09:21:31,2026-04-10 09:45:51,1,1,0,2,1647.0,...,4,morning,2,8,2,15.63,1.40,12.02,24.33,53.38
1,2026-04-10 08:48:56,2026-04-10 08:56:53,2026-04-10 08:59:36,2026-04-10 09:05:50,2026-04-10 09:16:38,1,0,0,1,1600.0,...,4,morning,2,7,2,7.95,2.72,6.23,10.80,27.70
2,2026-04-10 08:47:31,2026-04-10 08:57:08,2026-04-10 09:10:46,2026-04-10 09:17:52,2026-04-10 09:37:11,1,1,1,3,571.0,...,4,morning,2,6,1,9.62,13.63,7.10,19.32,49.67
3,2026-04-10 08:42:54,2026-04-10 08:44:29,2026-04-10 08:54:55,2026-04-10 09:10:20,2026-04-10 09:18:27,1,1,0,2,290.0,...,4,morning,2,7,0,1.58,10.43,15.42,8.12,35.55
4,2026-04-10 08:29:59,2026-04-10 08:35:23,2026-04-10 08:40:03,2026-04-10 08:49:56,2026-04-10 09:03:48,1,0,0,1,1600.0,...,4,morning,2,6,0,5.40,4.67,9.88,13.87,33.82
5,2026-04-10 08:18:45,2026-04-10 08:34:02,2026-04-10 08:36:06,2026-04-10 08:46:44,2026-04-10 09:02:12,1,1,0,2,1216.0,...,4,morning,2,9,0,15.28,2.07,10.63,15.47,43.45
6,2026-04-10 08:13:24,2026-04-10 08:36:00,2026-04-10 08:37:28,2026-04-10 08:58:18,2026-04-10 09:31:07,1,1,1,3,4302.0,...,4,morning,2,10,1,22.60,1.47,20.83,32.82,77.72
7,2026-04-10 07:58:25,2026-04-10 07:59:45,2026-04-10 08:03:30,2026-04-10 08:04:23,2026-04-10 08:15:08,1,0,0,1,360.0,...,4,morning,2,9,0,1.33,3.75,0.88,10.75,16.72
8,2026-04-10 07:44:48,2026-04-10 07:52:06,2026-04-10 08:08:27,2026-04-10 08:30:41,2026-04-10 09:05:24,1,1,0,2,3304.0,...,4,morning,2,8,3,7.30,16.35,22.23,34.72,80.60
9,2026-04-10 07:35:54,2026-04-10 07:52:57,2026-04-10 08:04:12,2026-04-10 08:17:39,2026-04-10 08:20:13,1,1,0,2,1090.0,...,4,morning,2,7,1,17.05,11.25,13.45,2.57,44.32


## Save Feature Data

บันทึกผลลัพธ์หลังสร้างฟีเจอร์แล้วไปยังโฟลเดอร์ `data/interim`


In [ ]:
output_path = INTERIM_DATA_DIR / 'vw_timestamp_dashboard_featured.csv'

try:
    df_feature.to_csv(output_path, index=False, encoding='utf-8-sig')
    saved_path = output_path
except PermissionError:
    saved_path = output_path.with_name(f'{output_path.stem}_latest.csv')
    df_feature.to_csv(saved_path, index=False, encoding='utf-8-sig')
    print(f'Could not overwrite {output_path.name}; saved fallback file instead.')

saved_path

WindowsPath('D:/ML/data/interim/vw_timestamp_dashboard_featured.csv')